In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

class ClassificationDataset:
    """
    Manages the FashionMNIST dataset for classification.
    Downloads the data on first use, applies ToTensor() preprocessing,
    and returns DataLoader objects for training and testing.
    """

    def __init__(self, batch_size=64):
        """
        Arguments:
            batch_size (int): number of images per batch (default: 64)
        """
        self.batch_size = batch_size

        # ToTensor() converts PIL images to float tensors in [0,1]
        # and reshapes from (H, W, C) to (C, H, W)
        self.transform = transforms.Compose([transforms.ToTensor()])

    def get_loaders(self):
        """
        Downloads FashionMNIST if not already present, and returns
        a DataLoader for the training split and one for the test split.

        Returns:
            train_loader: DataLoader over 60,000 training images
            test_loader:  DataLoader over 10,000 test images
        """
        # Download and wrap training split
        training_data = datasets.FashionMNIST(
            root="./data", train=True, download=True, transform=self.transform
        )

        # Download and wrap test split
        test_data = datasets.FashionMNIST(
            root="./data", train=False, download=True, transform=self.transform
        )

        # shuffle=True for training: randomizes batch order each epoch
        # shuffle=False for testing: order does not affect evaluation
        train_loader = DataLoader(training_data, batch_size=self.batch_size, shuffle=True)
        test_loader  = DataLoader(test_data,     batch_size=self.batch_size, shuffle=False)

        return train_loader, test_loader

In [ ]:
import torch
import torch.nn as nn

class LinearClassificationModel(nn.Module):
    """
    Linear classification model for FashionMNIST.
    Accepts raw image tensors of shape (batch_size, 1, 28, 28).
    Returns logits of shape (batch_size, output_classes).
    """
    def __init__(self, input_features, output_classes, sigma=0.01):
        super(LinearClassificationModel, self).__init__()

        # nn.Flatten() + nn.Linear: the minimal architecture for linear classification
        self.net = nn.Sequential(
            nn.Flatten(),                               # (batch, 1, 28, 28) → (batch, 784)
            nn.Linear(input_features, output_classes)   # (batch, 784) → (batch, 10)
        )

        # Weight initialization: N(0, sigma^2) for weights, zeros for biases
        nn.init.normal_(self.net[1].weight, mean=0, std=sigma)
        nn.init.zeros_(self.net[1].bias)

    def forward(self, X):
        """
        X: tensor of shape (batch_size, 1, 28, 28)
        Returns: logits of shape (batch_size, output_classes)
        """
        return self.net(X)

In [ ]:
class CrossEntropyLossWrapper:
    def __init__(self):
        self.criterion = nn.CrossEntropyLoss()
    
    def __call__(self, predictions, targets):
        return self.criterion(predictions, targets)

In [ ]:

class SGDOptimizer:
    """
    Wrapper class for SGD optimizer.
    """
    def __init__(self, model_params, lr=0.01):
        self.optimizer = torch.optim.SGD(model_params, lr=lr)
    
    def zero_grad(self):
        self.optimizer.zero_grad()
    
    def step(self):
        self.optimizer.step()

In [ ]:
############################################################
# Solving FashionMNIST with LinearClassificationModel
############################################################

# Step 1: Data preparation
data_manager = ClassificationDataset(batch_size=64)
train_loader, test_loader = data_manager.get_loaders()

# Step 2: Define model, loss function, and optimizer
input_size  = 28 * 28   # FashionMNIST images are 28x28 pixels
num_classes = 10        # 10 clothing categories

model     = LinearClassificationModel(input_size, num_classes)
loss_fn   = CrossEntropyLossWrapper()
optimizer = SGDOptimizer(model.parameters(), lr=0.01)

# Step 3: Training loop
epochs = 5
total_epoch_loss = []
final_accuracy   = []

model.train()

for epoch in range(epochs):

    total_loss = 0.0
    correct    = 0
    total      = 0

    for images, labels in train_loader:

        optimizer.zero_grad()              # 1. clear gradients
        outputs = model(images)            # 2. forward pass → logits (batch, 10)
        loss    = loss_fn(outputs, labels) # 3. compute cross-entropy loss
        loss.backward()                    # 4. compute gradients
        optimizer.step()                   # 5. update weights

        # accumulate metrics
        predicted   = torch.argmax(outputs, dim=1)
        correct    += (predicted == labels).sum().item()
        total      += labels.size(0)
        total_loss += loss.item()

    epoch_loss     = total_loss / len(train_loader)
    epoch_accuracy = 100. * correct / total

    total_epoch_loss.append(epoch_loss)
    final_accuracy.append(epoch_accuracy)

    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f} | Accuracy: {epoch_accuracy:.2f}%")

# Step 4: Evaluation on the test set
model.eval()

correct = 0
total   = 0

with torch.no_grad():
    for images, labels in test_loader:

        outputs   = model(images)                    # forward pass → logits
        predicted = torch.argmax(outputs, dim=1)     # predicted class per image

        correct += (predicted == labels).sum().item()
        total   += labels.size(0)

test_accuracy = 100 * correct / total
print(f"\nTest accuracy on 10,000 images: {test_accuracy:.2f}%")